In [2]:
import pandas as pd
import numpy as np
from library.connectors.openalex.openalex_connector import OpenAlexConnector

In [3]:
with open('../../data/openalex_ids_0812_final_24M.txt', 'r') as f:
    openalex_ids = [line.strip() for line in f]

In [4]:
sample_ids = np.random.choice(openalex_ids, size=1000, replace=False).tolist()
print(len(sample_ids))
sample_ids[:10]

1000


['W4315651312',
 'W4295693654',
 'W2796119270',
 'W2789941571',
 'W2231596758',
 'W4241967315',
 'W2621516501',
 'W2395811517',
 'W4230084066',
 'W1608313311']

In [9]:
works = OpenAlexConnector().get_works_from_ids(sample_ids, filters={}, fields=['id', 'title', 'abstract_inverted_index'])

In [10]:
res = [w for w in works]

In [21]:
res[2]['abstract']

"The development of shared memories, beliefs, and norms is a fundamental characteristic of human communities. These emergent outcomes are thought to occur owing to a dynamic system of information sharing and memory updating, which fundamentally depends on communication. Here we report results on the formation of collective memories in laboratory-created communities. We manipulated conversational network structure in a series of real-time, computer-mediated interactions in fourteen 10-member communities. The results show that mnemonic convergence, measured as the degree of overlap among community members' memories, is influenced by both individual-level information-processing phenomena and by the conversational social network structure created during conversational recall. By studying laboratory-created social networks, we show how large-scale social phenomena (i.e., collective memory) can emerge out of microlevel local dynamics (i.e., mnemonic reinforcement and suppression effects). Th

In [16]:
res[0]['abstract']

In [26]:
wdf = pd.DataFrame(res)

In [27]:
wdf['abstract'] = [w['abstract'] for w in res]

In [30]:
wdf = wdf[['title', 'abstract']].dropna()

In [37]:
wdf = wdf[wdf.abstract.str.len() > 100]

In [39]:
from setfit import SetFitModel

In [42]:
model = SetFitModel.from_pretrained(
    "TheoLvs/wsl-prescreening-multi-v0.0",
    multi_target_strategy="multi-output",
)

In [43]:
model.predict_proba(["This is a test sentence."])

tensor([[[0.7354, 0.2646],
         [0.9243, 0.0757],
         [0.9127, 0.0873],
         [0.9622, 0.0378],
         [0.8946, 0.1054]]], dtype=torch.float64)

In [44]:
def get_preds(df, model, batch_size=100):
    preds = model.predict_proba(
        df["abstract"].to_list(), batch_size=batch_size, show_progress_bar=True
    )
    labelnames = ["other", "planetary_boundaries", "well_being", "resources", "justice"]
    cols = ["proba_" + x for x in labelnames]
    preds_df = pd.DataFrame(preds[:, :, 1].numpy(), columns=cols)
    return pd.concat((df.reset_index(drop=True), preds_df), axis=1)

In [45]:
preds = get_preds(wdf, model)

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

In [48]:
def add_screen_class(df):
    preds_df = df[
        [
            "proba_other",
            "proba_planetary_boundaries",
            "proba_well_being",
            "proba_resources",
            "proba_justice",
        ]
    ]
    prescreening_high = preds_df.idxmax(axis=1) != "proba_other"
    pred_class = preds_df.idxmax(axis=1).map(lambda x: x.replace("proba_", ""))

    prescreening_medium = prescreening_high.copy()
    prescreening_medium.loc[preds_df.max(axis=1) < 0.5] = False

    prescreening_low = prescreening_high.copy()
    prescreening_low.loc[preds_df.max(axis=1) < 0.8] = False

    df["prescreening_high"] = prescreening_high
    df["prescreening_medium"] = prescreening_medium
    df["prescreening_low"] = prescreening_low
    df["pred_class"] = pred_class

    return df

In [52]:
final = add_screen_class(preds)
final.head()

,title,abstract,proba_other,proba_planetary_boundaries,proba_well_being,proba_resources,proba_justice,prescreening_high,prescreening_medium,prescreening_low,pred_class
0,Mnemonic convergence in social networks: The e...,"The development of shared memories, beliefs, a...",0.852960,0.034826,0.039479,0.037360,0.035747,False,False,False,other
1,Prosthetic ankle push-off work reduces metabol...,Individuals with unilateral below-knee amputat...,0.045939,0.051595,0.829662,0.033199,0.025903,True,True,True,well_being
2,The Use of a Decision Support System for Susta...,The increasing level of antropopression has a ...,0.043277,0.822457,0.042055,0.040947,0.039247,True,True,True,planetary_boundaries
3,Surviving in a toxic world: transcriptomics an...,Together with the induced high expression of d...,0.149324,0.239474,0.344471,0.018327,0.024119,True,False,False,well_being
4,xCos: An Explainable Cosine Metric for Face Ve...,We study the XAI (explainable AI) on the face ...,0.853580,0.035214,0.037835,0.038561,0.035597,False,False,False,other


In [80]:
import asyncio
import time
from typing import Literal
from google import genai
from dotenv import load_dotenv
from pydantic import BaseModel
from tqdm.auto import tqdm
load_dotenv()

True

In [56]:
client = genai.Client()

In [93]:
class PrescreeningResponse(BaseModel):
    category: Literal[
        "planetary_boundaries",
        "well_being",
        "resources",
        "justice",
        "other",
    ]
    relevance: Literal["high", "medium", "low"]

In [94]:
system_prompt = """
Classify the following scientific abstract into one of the following topics: planetary boundaries, wellbeing, natural resources, justice, other.
Also classify the relevance of the abstract to the topic as high, medium or low.
"""

In [95]:
async def extract_topic(
    text: str, prompt: str, model_name: str, client: genai.Client = client
) -> PrescreeningResponse:
    try:
        def _call():
            return client.models.generate_content(
                model=model_name,
                contents=text,
                config=genai.types.GenerateContentConfig(
                    system_instruction=prompt.strip(),
                    thinking_config=genai.types.ThinkingConfig(thinking_level="minimal"),
                    temperature=0,
                    max_output_tokens=256,
                    response_mime_type="application/json",
                    response_schema=PrescreeningResponse,
                ),
            )
        response = await asyncio.to_thread(_call)
        return PrescreeningResponse.model_validate_json(response.text)
    except Exception as e:
        print('Error')
        if 'response' in locals():
            print(response.text)
        return f"Error: {e}"


async def batch_extract_topic(
    texts: list[str], prompt: str, model_name: str, client: genai.Client = client
) -> list[PrescreeningResponse]:
    results = await asyncio.gather(*[extract_topic(text, prompt, model_name, client) for text in texts])
    return results


In [96]:
model_name = "gemini-3-flash-preview"

In [97]:
res = await extract_topic(wdf.abstract.iloc[1], system_prompt, model_name, client)
res

PrescreeningResponse(category='well_being', relevance='low')

In [98]:
# process by batch
batch_size = 20
rpm_quota = 1000
wait_time = 60 / (rpm_quota / batch_size)
results = []
for i in tqdm(range(0, len(wdf), batch_size)):
    batch = wdf.iloc[i:i+batch_size]
    texts = batch.abstract.values.tolist()
    preds = await batch_extract_topic(texts, system_prompt, model_name, client)
    results.extend(preds)
    time.sleep(wait_time)

  0%|          | 0/40 [00:00<?, ?it/s]

In [99]:
final['gemini_pred'] = [r.category for r in results]
final['gemini_kept'] = final['gemini_pred'] != 'other'
final.head()

,title,abstract,proba_other,proba_planetary_boundaries,proba_well_being,proba_resources,proba_justice,prescreening_high,prescreening_medium,prescreening_low,pred_class,gemini_pred,gemini_kept
0,Mnemonic convergence in social networks: The e...,"The development of shared memories, beliefs, a...",0.852960,0.034826,0.039479,0.037360,0.035747,False,False,False,other,other,False
1,Prosthetic ankle push-off work reduces metabol...,Individuals with unilateral below-knee amputat...,0.045939,0.051595,0.829662,0.033199,0.025903,True,True,True,well_being,well_being,True
2,The Use of a Decision Support System for Susta...,The increasing level of antropopression has a ...,0.043277,0.822457,0.042055,0.040947,0.039247,True,True,True,planetary_boundaries,planetary_boundaries,True
3,Surviving in a toxic world: transcriptomics an...,Together with the induced high expression of d...,0.149324,0.239474,0.344471,0.018327,0.024119,True,False,False,well_being,other,False
4,xCos: An Explainable Cosine Metric for Face Ve...,We study the XAI (explainable AI) on the face ...,0.853580,0.035214,0.037835,0.038561,0.035597,False,False,False,other,other,False


In [114]:
final[['proba_other', 'proba_planetary_boundaries', 'proba_well_being', 'proba_resources', 'proba_justice']].sum(axis=1)

0      1.000372
1      0.986296
2      0.987984
3      0.775714
4      1.000786
         ...   
788    0.997931
789    0.972663
790    0.985471
791    0.992157
792    0.999277
Length: 793, dtype: float64

In [105]:
from sklearn.metrics import classification_report, confusion_matrix

In [101]:
print(classification_report(final['gemini_pred'], final['pred_class']))

                      precision    recall  f1-score   support

             justice       0.69      0.30      0.42        74
               other       0.69      0.77      0.72       405
planetary_boundaries       0.04      0.80      0.07        10
           resources       0.61      0.09      0.16       152
          well_being       0.69      0.32      0.43       152

            accuracy                           0.51       793
           macro avg       0.54      0.45      0.36       793
        weighted avg       0.66      0.51      0.52       793



In [109]:
# many natural resources are classified as planetary boundaries, which isn't really a problem for screening
print(confusion_matrix(final['gemini_pred'], final['pred_class'], labels=['planetary_boundaries', 'resources', 'other']))

[[  8   2   0]
 [115  14  23]
 [ 65   4 310]]


In [108]:
print(classification_report(final['gemini_kept'], final['prescreening_high']))
print('---')
print(classification_report(final['gemini_kept'], final['prescreening_medium']))
print('---')
print(classification_report(final['gemini_kept'], final['prescreening_low']))

              precision    recall  f1-score   support

       False       0.69      0.77      0.72       405
        True       0.72      0.64      0.68       388

    accuracy                           0.70       793
   macro avg       0.70      0.70      0.70       793
weighted avg       0.70      0.70      0.70       793

---
              precision    recall  f1-score   support

       False       0.67      0.82      0.74       405
        True       0.76      0.58      0.66       388

    accuracy                           0.70       793
   macro avg       0.71      0.70      0.70       793
weighted avg       0.71      0.70      0.70       793

---
              precision    recall  f1-score   support

       False       0.57      0.95      0.71       405
        True       0.83      0.24      0.37       388

    accuracy                           0.60       793
   macro avg       0.70      0.60      0.54       793
weighted avg       0.70      0.60      0.55       793

